# TEST SINGLE AUGMENTATION

In [ ]:
from pathlib import Path
import numpy as np
import os
import io
from PIL import Image
import torch

from emma import EmmaDataset, EmmaSearcher, SearchQuery
from transformers import Sam3Config, Sam3Processor, Sam3Model

import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Ready")
else:
    device = torch.device("cpu")
    print("Not Ready")

## > DATASET

In [ ]:
dataset_root = None  # e.g. Path("/path/to/EMMa")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the EMMa dataset.")

dataset_root = Path(dataset_root)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")

In [ ]:
# INIT DATASET
ds = EmmaDataset(dataset_root)

In [ ]:
# QUERY
se = EmmaSearcher(ds)
query = SearchQuery(splits=["Test"])
asins = se.query_asins(query)
print(len(asins))

In [ ]:
# SELECT A SAMPLE
rec = ds.get_record(asins[10])

title = ds.get_title(rec=rec)
categories = ds.get_category_names(rec=rec)

image_dict = ds.get_images(rec)
first_img_key, first_img_bytes = next(iter(image_dict.items()))
first_img = Image.open(io.BytesIO(first_img_bytes)).convert("RGB")

print("Title: ", title)
print("Category Path: ", categories)
print("Deepest Category: ", categories[-1])
first_img.show()

## > MODEL

In [ ]:
sam3_root = None  # e.g. Path("/path/to/SAM3")

if sam3_root is None:
    raise ValueError("Please set `sam3_root` to the path of the SAM3 model.")

sam3_root = Path(sam3_root)

if not sam3_root.exists():
    raise FileNotFoundError(f"Model path does not exist: {sam3_root}")

In [ ]:
RES = 1008  # try 560, 700, 840, 1008

config = Sam3Config.from_pretrained(sam3_root)
config.image_size = RES

model_kwargs = {
    "config": config,
    "use_safetensors": True,
}
if device.type == "cuda":
    model_kwargs["device_map"] = "cuda"
    model_kwargs["torch_dtype"] = torch.float16
else:
    model_kwargs["torch_dtype"] = torch.float32

segmentation_model = Sam3Model.from_pretrained(sam3_root, **model_kwargs).eval()
if device.type != "cuda":
    segmentation_model = segmentation_model.to(device)

segmentation_processor = Sam3Processor.from_pretrained(
    sam3_root,
    size={"height": RES, "width": RES},
)
print(f"Model loaded on {device}")

## > AUGMENTATION

In [ ]:
input_image = [first_img]

# Process image only
segmentation_inputs = segmentation_processor(
    images=input_image,
    return_tensors="pt",
)

# Tokenize title, forcing truncation to 32 tokens
text_inputs = segmentation_processor.tokenizer(
    title,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=32,
)

# # Tokenize deepest category
# text_inputs = segmentation_processor.tokenizer(
#     categories[-1],
#     return_tensors="pt",
# )

# Merge both inputs
segmentation_inputs.update(text_inputs)
segmentation_inputs = segmentation_inputs.to(device)

In [ ]:
def show_segmentation_results(image, result):
    if hasattr(image, "convert"):  # PIL image
        image = np.array(image.convert("RGB"))
    else:
        image = np.array(image)

    masks = result["masks"]
    boxes = result["boxes"]
    scores = result["scores"]

    if hasattr(masks, "detach"):
        masks = masks.detach().cpu().numpy()
    if hasattr(boxes, "detach"):
        boxes = boxes.detach().cpu().numpy()
    if hasattr(scores, "detach"):
        scores = scores.detach().cpu().numpy()

    if masks.ndim == 2:
        masks = masks[None, ...]
    elif masks.ndim == 4 and masks.shape[1] == 1:
        masks = masks[:, 0]

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image)

    cmap = plt.get_cmap("tab10")

    for i, (mask, box, score) in enumerate(zip(masks, boxes, scores)):
        color = np.array(cmap(i % 10)[:3])

        overlay = np.zeros((*mask.shape, 4))
        overlay[..., :3] = color
        overlay[..., 3] = (mask > 0.5) * 0.4
        ax.imshow(overlay)

        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)

        ax.text(x1, y1, f"{score:.2f}", color="white")

    ax.axis("off")
    plt.show()

In [ ]:
with torch.no_grad():
    segmentation_outputs = segmentation_model(**segmentation_inputs)

segmentation_result = segmentation_processor.post_process_instance_segmentation(
    segmentation_outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=segmentation_inputs["original_sizes"].tolist(),
)[0]

In [ ]:
show_segmentation_results(input_image[0], segmentation_result)
plt.show()